# 03. Advanced Modeling & Class Balancing
**Objective:** Address class imbalance using SMOTE and train advanced ensemble models (XGBoost).

### Step 1: Environment Setup & Cleaning Logic
We reuse the `clean_text` logic from the baseline phase to ensure consistency.

In [6]:
import pandas as pd
import numpy as np
import re
import spacy
import joblib
from nltk.sentiment import SentimentIntensityAnalyzer

# Load NLP tools
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
sia = SentimentIntensityAnalyzer()


def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Standardize: lowercase and remove special characters
    text = re.sub(r"[^a-zA-Z\s]", "", text.lower())
    # Process with spaCy
    doc = nlp(text)
    # Use .lemma_ to get the root word (e.g., "batteries" -> "battery")
    return " ".join(
        [token.lemma_ for token in doc if not token.is_stop and len(token.text) > 2]
    )


# Load the saved Vectorizer from Notebook 02
try:
    tfidf = joblib.load("../models/vectorizer.pkl")
    print("Vectorizer loaded successfully.")
except:
    print("Error: vectorizer.pkl not found in /models folder.")

Vectorizer loaded successfully.


### Step 2: Harmonizing Datasets
We load both datasets, identify the text columns, and generate sentiment labels using VADER.

In [7]:
# 1. Load files
df_amazon = pd.read_csv("../data/processed/amazon_cleaned.csv")
df_mendeley = pd.read_csv("../data/processed/mendeley_cleaned.csv")

# 2. Extract Text (Handle Amazon's 'name' vs Mendeley's 'Review')
df_amazon["raw_text"] = df_amazon["name"] if "name" in df_amazon.columns else ""
df_mendeley["raw_text"] = (
    df_mendeley["Review"] if "Review" in df_mendeley.columns else ""
)


# 3. Generate Sentiment Labels (VADER)
def get_vader_sentiment(text):
    score = sia.polarity_scores(str(text))["compound"]
    if score >= 0.05:
        return "positive"
    if score <= -0.05:
        return "negative"
    return "neutral"


print("Generating sentiment labels...")
df_amazon["sentiment"] = df_amazon["raw_text"].apply(get_vader_sentiment)
df_mendeley["sentiment"] = df_mendeley["raw_text"].apply(get_vader_sentiment)

# 4. Merge and Clean
df = pd.concat(
    [df_amazon[["raw_text", "sentiment"]], df_mendeley[["raw_text", "sentiment"]]],
    ignore_index=True,
)

print("Applying text cleaning (this may take a minute)...")
df["clean_text"] = df["raw_text"].apply(clean_text)

# Drop empty rows
df = df.dropna(subset=["clean_text", "sentiment"])
df = df[df["clean_text"].str.strip() != ""]

print(f"Data ready for training. Total rows: {len(df)}")
print("Class Distribution:\n", df["sentiment"].value_counts())

Generating sentiment labels...
Applying text cleaning (this may take a minute)...
Data ready for training. Total rows: 34151
Class Distribution:
 sentiment
positive    24256
neutral      8912
negative      983
Name: count, dtype: int64


### Step 3: Training the Advanced Model
We use SMOTE to balance the training set and XGBoost to perform the classification.

In [9]:
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Encode Labels
le = LabelEncoder()
y_encoded = le.fit_transform(df["sentiment"])
X_features = tfidf.transform(df["clean_text"])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Apply SMOTE to the training set only
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

# 1. Train XGBoost (variable named 'xgb')
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric="mlogloss")
xgb.fit(X_resampled, y_resampled)

# 2. Test and Report
y_pred = xgb.predict(X_test)
print("--- Advanced Model (XGBoost + SMOTE) Performance ---")
print(classification_report(y_test, y_pred, target_names=le.classes_))


/home/skylar_lorena/anaconda3/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [18:44:05] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


--- Advanced Model (XGBoost + SMOTE) Performance ---
              precision    recall  f1-score   support

    negative       0.64      0.65      0.64       197
     neutral       0.82      0.98      0.89      1782
    positive       0.99      0.92      0.95      4852

    accuracy                           0.93      6831
   macro avg       0.82      0.85      0.83      6831
weighted avg       0.94      0.93      0.93      6831



In [ ]:

# 3. Save the model and label encoder
import joblib
import os

# Create directory if it doesn't exist
os.makedirs("../models", exist_ok=True)

joblib.dump(xgb, "../models/advanced_model_xgb.pkl")
joblib.dump(le, "../models/label_encoder.pkl")

print("Notebook 03 complete. Assets saved to /models.")

Notebook 03 complete. Assets saved to /models.
